# 03 · Modeling

Every guess and model is tested season by season with `src/evaluation.py`: for each of the last 9 seasons (2000–2009, no 2004), it may only learn from the seasons before, then predicts that season. The score is the average over the 9 seasons.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.evaluation import test_seasons, evaluate_by_season, division_rank, mean_rank_error, pick_with_practice_test

team_table = pd.read_csv("../data/team_table.csv")
seasons = test_seasons(team_table)
seasons

## Task (a): ranking

Score: on average, how many positions off the predicted rank is (0 = perfect).

### Simple guess: same order as last season

Inside each division, the team with the most points per game last season is predicted 1st, the next 2nd, and so on. New teams go last. This is "same position as last season" when divisions stay the same, and still works when they change (e.g. 1998, when the league went from 4 to 6 divisions).

In [ ]:
def same_order_as_last_season(train, test):
    return division_rank(test, test["last_pts_vs_avg"])

simple_a = evaluate_by_season(team_table, same_order_as_last_season, mean_rank_error, seasons)
print("Average positions off:", round(simple_a["score"].mean(), 2))
simple_a

### For comparison: random order

What a guess with no information scores. Any model has to be clearly better than this, and better than the simple guess above.

In [ ]:
rng = np.random.default_rng(0)

def random_order(train, test):
    return division_rank(test, rng.random(len(test)))

random_a = evaluate_by_season(team_table, random_order, mean_rank_error, seasons)
print("Average positions off:", round(random_a["score"].mean(), 2))

### First models (first try)

Each model learns, from the training seasons only, a team's **relative position** in its division (`rank / div_size`, so 1st of 5 and 1st of 7 are comparable). Its prediction is then turned into positions 1, 2, 3… inside each division with `division_rank`.

- **Linear regression, 3 features:** last season's points and the two roster columns.
- **Linear regression, all features.**
- **Random forest, all features:** many decision trees averaged; can pick up non-linear patterns.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

ALL_FEATURES = ["last_pts_vs_avg", "last_gf_vs_avg", "last_ga_vs_avg", "last_made_playoffs", "last_rank",
                "new_team", "roster_pts_vs_avg", "roster_goalie_ga_vs_avg", "div_size"]
FEW_FEATURES = ["last_pts_vs_avg", "roster_pts_vs_avg", "roster_goalie_ga_vs_avg"]

def model_predict(make_model, features):
    """A predict(train, test) function for the test loop: learn relative position, return ranks."""
    def predict(train, test):
        model = make_model().fit(train[features], train["rank"] / train["div_size"])
        return division_rank(test, -model.predict(test[features]))   # lower predicted position = better
    return predict

models = {
    "Linear, 3 features": model_predict(LinearRegression, FEW_FEATURES),
    "Linear, all features": model_predict(LinearRegression, ALL_FEATURES),
    "Random forest, all features": model_predict(
        lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0), ALL_FEATURES),
}
results = {"Random order": random_a, "Simple guess": simple_a}
for name, predict in models.items():
    results[name] = evaluate_by_season(team_table, predict, mean_rank_error, seasons)

summary = pd.DataFrame({name: r.set_index("season")["score"] for name, r in results.items()})
print(summary.mean().round(2).sort_values())
summary.round(2)

### Fair choice of settings (practice test)

In the first try we picked the best version **after** seeing the test scores, which makes it look slightly better than it is. Here each version is chosen **inside the training seasons only**: the last 3 training seasons are a practice test, the version with the best practice score is chosen, and only then is it scored on the real test season.

- **Linear:** choose between 3 features and all features.
- **Random forest:** choose how detailed the trees are: at least 5, 20 or 40 teams per leaf, and at most 3, 6 or unlimited questions in a row. Less detailed trees cannot memorise the training seasons.

This cell takes about a minute.

In [ ]:
forest_candidates = {
    f"leaf {leaf}, depth {depth}": model_predict(
        lambda leaf=leaf, depth=depth: RandomForestRegressor(n_estimators=200, min_samples_leaf=leaf, max_depth=depth,
                                                             random_state=0, n_jobs=-1), ALL_FEATURES)
    for leaf in [5, 20, 40] for depth in [3, 6, None]
}
linear_candidates = {
    "3 features": model_predict(LinearRegression, FEW_FEATURES),
    "all features": model_predict(LinearRegression, ALL_FEATURES),
}

chosen = {}   # which version was chosen for each test season

def choose_then_predict(candidates, label):
    def predict(train, test):
        best, _ = pick_with_practice_test(train, candidates, mean_rank_error)
        chosen.setdefault(label, []).append(best)
        return candidates[best](train, test)
    return predict

fair = {
    "Simple guess": simple_a,
    "Linear, chosen fairly": evaluate_by_season(team_table, choose_then_predict(linear_candidates, "Linear"), mean_rank_error, seasons),
    "Forest, chosen fairly": evaluate_by_season(team_table, choose_then_predict(forest_candidates, "Forest"), mean_rank_error, seasons),
}
fair_summary = pd.DataFrame({name: r.set_index("season")["score"] for name, r in fair.items()})
print(fair_summary.mean().round(2))
print()
print(pd.DataFrame(chosen, index=seasons))
fair_summary.round(2)

### Checks: is the difference real, and does a model memorise?

- **Better / equal / worse:** in how many of the 9 seasons the model beat, matched or lost to the simple guess.
- **Wilcoxon p-value:** how often a difference this big would appear by chance if the model were no better. Below 0.05 counts as real.
- **Training vs test error:** if the error on the seasons a model learned from is much lower than on new seasons, it memorised them (overfitting).

In [ ]:
from scipy.stats import wilcoxon

def training_error(predict):
    """Average error of a model on the seasons it learned from (same 9 refits as the test loop)."""
    return np.mean([mean_rank_error(team_table[team_table["year"] < s],
                                    predict(team_table[team_table["year"] < s], team_table[team_table["year"] < s]))
                    for s in seasons])

checks = {}
for name, r in fair.items():
    if name == "Simple guess":
        continue
    diff = r["score"] - simple_a["score"]
    checks[name] = {"better": int((diff < 0).sum()), "equal": int((diff == 0).sum()), "worse": int((diff > 0).sum()),
                    "Wilcoxon p": round(wilcoxon(r["score"], simple_a["score"]).pvalue, 2),
                    "test error": round(r["score"].mean(), 2)}

# Training error: the first-try forest vs the less detailed forest chosen most often vs linear
checks["Linear, chosen fairly"]["training error"] = round(training_error(linear_candidates["3 features"]), 2)
checks["Forest, chosen fairly"]["training error"] = round(training_error(forest_candidates["leaf 20, depth 3"]), 2)
first_try_forest = model_predict(lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0), ALL_FEATURES)
print("First-try forest (leaf 5, unlimited depth): training error", round(training_error(first_try_forest), 2),
      "vs test error", round(results["Random forest, all features"]["score"].mean(), 2))
pd.DataFrame(checks).T

### What this shows

- The linear model with 3 features is the best (1.02 positions off vs 1.07 for the simple guess), and the practice test chooses it in every season, so the number is honest.
- No model is proven better than the simple guess (Wilcoxon p above 0.05): last season already says most of what can be known before the season, and 9 test seasons are too few to prove a small gap.
- The first-try forest memorised the training seasons (overfitting: 0.57 vs 1.07). Less detailed trees stop that, but still do not beat the simple guess.

Plain-language explanation: `docs/modeling_explained.md`.

### New inputs (from notebook 02)

We test whether the three new inputs help, letting the **practice test** choose fairly between versions of the 3-feature linear model: with nothing added, with one of the new inputs, or with all three. The version can change from season to season, because it is chosen again before each test season using only the seasons before it.

In [ ]:
from scipy.stats import wilcoxon

input_candidates = {
    "3 features": model_predict(LinearRegression, FEW_FEATURES),
    "+ 3-season average": model_predict(LinearRegression, FEW_FEATURES + ["avg3_pts_vs_avg"]),
    "+ roster change": model_predict(LinearRegression, FEW_FEATURES + ["roster_change_vs_avg"]),
    "+ new coach": model_predict(LinearRegression, FEW_FEATURES + ["new_coach"]),
    "+ all three": model_predict(LinearRegression, FEW_FEATURES + ["avg3_pts_vs_avg", "roster_change_vs_avg", "new_coach"]),
}

chosen_inputs = []

def choose_inputs(train, test):
    best, _ = pick_with_practice_test(train, input_candidates, mean_rank_error)
    chosen_inputs.append(best)
    return input_candidates[best](train, test)

new_inputs = evaluate_by_season(team_table, choose_inputs, mean_rank_error, seasons)
diff = new_inputs["score"] - simple_a["score"]
print("Average positions off:", round(new_inputs["score"].mean(), 3))
print("Better / equal / worse than the simple guess:", int((diff < 0).sum()), int((diff == 0).sum()), int((diff > 0).sum()))
print("Wilcoxon p:", round(wilcoxon(new_inputs["score"], simple_a["score"]).pvalue, 2))
pd.DataFrame({"version chosen": chosen_inputs,
              "positions off": new_inputs["score"].round(2).values,
              "simple guess": simple_a["score"].round(2).values}, index=seasons)

### What the new inputs show

- With the new inputs available, the model reaches **0.985** positions off: the first result below 1, better than the simple guess in 6 seasons, equal in 2, worse in 1.
- The Wilcoxon p is 0.08: closer, but still above 0.05, so the gain is **not yet proven** to be more than luck.
- The practice test mostly chose **+ new coach** (5 seasons) and **+ 3-season average** (3 seasons). **Roster change** was never chosen: it barely relates to the final position.
- A new coach and a steadier, multi-season view of the team add a little information beyond last season alone.

### More model types

Four model types with different ways of learning, all using the same 5 inputs (the 3 features plus the 3-season average and new coach) and all with their settings chosen by the practice test:

- **Decision tree:** one chain of yes/no questions (e.g. *last season's points above 1.09?*). Easy to read; we choose how many questions in a row (depth 2–4) and how many teams each final answer needs (10 or 30).
- **k-nearest neighbours:** predicts a team from the *k* most similar past team-seasons. The inputs are first put on the same scale (`StandardScaler`), otherwise the input with the biggest numbers would decide what "similar" means. We choose *k* (10, 25, 50, 100).
- **Gradient boosting:** many small trees built one after another, each correcting the mistakes of the previous ones. We choose the number of trees (50 or 150) and their depth (2 or 3).
- **Linear:** the weighted sum from before, with the same 5 inputs fixed, for comparison.

This cell takes about 15 seconds.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from pathlib import Path
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)

# The inputs the practice test chose for the linear model: the 3 features plus the 3-season average and new coach
BEST_FEATURES = FEW_FEATURES + ["avg3_pts_vs_avg", "new_coach"]

chosen_models = {}   # which setting was chosen for each test season

def chosen_fairly(candidates, label):
    """A predict function that picks one candidate per test season with the practice test."""
    def predict(train, test):
        best, _ = pick_with_practice_test(train, candidates, mean_rank_error)
        chosen_models.setdefault(label, []).append(best)
        return candidates[best](train, test)
    return predict

model_candidates = {
    "Decision tree": {
        f"depth {d}, leaf {l}": model_predict(lambda d=d, l=l: DecisionTreeRegressor(max_depth=d, min_samples_leaf=l, random_state=0), BEST_FEATURES)
        for d in [2, 3, 4] for l in [10, 30]
    },
    "k-nearest neighbours": {
        f"k = {k}": model_predict(lambda k=k: make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k)), BEST_FEATURES)
        for k in [10, 25, 50, 100]
    },
    "Gradient boosting": {
        f"{n} trees, depth {d}": model_predict(lambda n=n, d=d: GradientBoostingRegressor(n_estimators=n, max_depth=d, learning_rate=0.05,
                                                                                     subsample=0.8, random_state=0), BEST_FEATURES)
        for n in [50, 150] for d in [2, 3]
    },
    "Linear, 5 inputs": {"5 inputs": model_predict(LinearRegression, BEST_FEATURES)},
}

all_models = {"Random order": random_a, "Simple guess": simple_a}
for name, candidates in model_candidates.items():
    all_models[name] = evaluate_by_season(team_table, chosen_fairly(candidates, name), mean_rank_error, seasons)

### Comparison of all models

One row per model: test error (positions off), seasons better / equal / worse than the simple guess, Wilcoxon p (chance the difference is luck), training error (error on the seasons it learned from) and the setting the practice test chose most often.

In [ ]:
def most_chosen(label):
    return pd.Series(chosen_models[label]).value_counts().index[0]

rows = {}
for name, r in all_models.items():
    row = {"test error": round(r["score"].mean(), 3)}
    if name in model_candidates:
        diff = r["score"] - simple_a["score"]
        row.update({"better": int((diff < 0).sum()), "equal": int((diff == 0).sum()), "worse": int((diff > 0).sum()),
                    "Wilcoxon p": round(wilcoxon(r["score"], simple_a["score"]).pvalue, 2),
                    "training error": round(training_error(model_candidates[name][most_chosen(name)]), 3),
                    "most chosen setting": most_chosen(name)})
    rows[name] = row
comparison = pd.DataFrame(rows).T.sort_values("test error")
comparison

### The decision tree, drawn

A depth-3 tree trained on all seasons, to show how it decides. Each box: the question, how many team-seasons reach it (`samples`) and their average relative position (`value`, 0 = top, 1 = bottom). Saved as `figures/decision_tree_a.png`.

In [ ]:
tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=30, random_state=0)
tree.fit(team_table[BEST_FEATURES], team_table["rank"] / team_table["div_size"])
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(tree, feature_names=BEST_FEATURES, filled=True, rounded=True, precision=2, impurity=False, fontsize=10, ax=ax)
ax.set_title("Decision tree for the relative position in the division (0 = top, 1 = bottom), NHL 1979-2009")
plt.savefig(FIG / "decision_tree_a.png", dpi=150, bbox_inches="tight")
plt.show()

### Which inputs matter (feature importance)

For the 5-input linear model, in every test season: we **scramble** one input (shuffle its values between teams) and measure how many more positions off the prediction becomes. A big increase means the model relies on that input; around zero means it does not help. Averaged over 20 scrambles and the 9 test seasons. Saved as `figures/feature_importance_a.png`.

In [ ]:
rng_imp = np.random.default_rng(0)
importance = {f: [] for f in BEST_FEATURES}
for s in seasons:
    train, test = team_table[team_table["year"] < s], team_table[team_table["year"] == s]
    model = LinearRegression().fit(train[BEST_FEATURES], train["rank"] / train["div_size"])
    def error(t):
        return mean_rank_error(t, division_rank(t, -model.predict(t[BEST_FEATURES])))
    base_error = error(test)
    for f in BEST_FEATURES:
        worse = []
        for _ in range(20):                       # scramble 20 times and average, to reduce luck
            scrambled = test.copy()
            scrambled[f] = rng_imp.permutation(scrambled[f].values)
            worse.append(error(scrambled) - base_error)
        importance[f].append(np.mean(worse))
importance = pd.Series({f: np.mean(v) for f, v in importance.items()}).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
importance.plot(kind="barh", ax=ax, color="tab:blue")
ax.set(xlabel="extra positions off when this input is scrambled", ylabel="",
       title="How much each input helps the linear model (test seasons 2000-2009)")
plt.tight_layout()
plt.savefig(FIG / "feature_importance_a.png", dpi=150)
plt.show()
print(importance.round(3).sort_values(ascending=False))

### What the models show

- **No model type beats the linear one**, and none is proven better than the simple guess (all Wilcoxon p well above 0.05). Linear with the 5 inputs fixed: 1.04; k-nearest neighbours and gradient boosting: 1.07, like the simple guess; the decision tree is worse (1.14).
- **Gradient boosting** has a training error of 0.84 vs 1.07 on new seasons: it still memorises part of the training seasons, like the first forest.
- **The decision tree** reads like hockey: first it splits on last season's points (above or below 1.09 of the average), then on the 3-season average and the goalies. Teams strong both last season and over 3 seasons end up highest (value 0.27–0.39); weak teams with leaky goalies lowest (0.91).
- **Importance:** last season's points matter most (scrambling them adds 0.22 positions off), then the 3-season average (0.09); the roster columns add a little (0.02–0.03) and new coach nothing (−0.02, scrambling it even helps slightly).
- **The 0.985 from the fair choice of inputs is fragile:** with a fixed set of 5 inputs the linear model scores 1.04, so the gain came from switching inputs from season to season. The solid findings are: last season's points first, the 3-season average second, everything else close to nothing.

## Task (b): coach change

Same loop: `evaluate_by_season(table, predict, score, seasons)`.

- `predict(train, test)` returns 1 (new coach) or 0 for each row of `test`, learning only from `train`.
- `score(test, predictions)` returns one number, e.g. the share of real coach changes caught (recall).
- To choose between versions fairly, use `pick_with_practice_test` (see task (a) above).